# R05 · Yocto I - layers, recipes and how BitBake evaluates variables

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R05_yocto_i_layers_recipes_and_how_bitbake_evaluates.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 70 min

**Skill Hub concepts:** Yocto Project and Poky; BitBake and tasks; Recipe (.bb) and append file (.bbappend); Layer; Board support package layer and machine configuration; Image recipe and IMAGE_INSTALL; Distribution features and init system

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R05")

## Map: what goes into a Yocto build

```
 Poky = BitBake (task executor) + OpenEmbedded-Core (meta) + meta-poky (reference distro) + meta-yocto-bsp

 conf/bblayers.conf: meta, meta-poky, meta-openembedded/meta-oe, meta-st-stm32mp (BSP), meta-rdl (yours)
 conf/local.conf:    MACHINE = "stm32mp1-disco"   DISTRO = "poky"   (your build settings)

 layer ─ conf/layer.conf (BBFILE_PRIORITY, LAYERSERIES_COMPAT)
       ├ conf/machine/<machine>.conf    BSP: CPU tuning, kernel provider, device tree, serial console
       ├ conf/distro/<distro>.conf      DISTRO_FEATURES (systemd, wifi, x11...), init manager
       ├ recipes-*/<pkg>/<pkg>_<ver>.bb  how to fetch, configure, compile, install one piece of software
       ├ recipes-*/<pkg>/<pkg>_%.bbappend   your changes to someone else's recipe, without editing it
       └ recipes-core/images/<image>.bb  IMAGE_INSTALL = which packages end up in the root filesystem

 per recipe: do_fetch -> do_unpack -> do_patch -> do_configure -> do_compile -> do_install -> do_package -> ... -> image
```

**How BitBake evaluates a variable** (the part interviews and debugging both hit):

| Syntax | When it applies | Effect |
|---|---|---|
| `A = "x"` | parse time | set, **expanded lazily** (`${B}` resolved when A is read) |
| `A := "x"` | parse time | set, expanded **now** (references to undefined variables stay literal) |
| `A ?= "x"` | parse time | set only if A has no value yet (weak defaults do not count) |
| `A ??= "x"` | end of parsing | weak default, used only if nothing else set A; the **last** `??=` wins |
| `A += "x"` / `A =+ "x"` | parse time | append / prepend **with** a space |
| `A .= "x"` / `A =. "x"` | parse time | append / prepend **without** a space |
| `A:append = "x"` / `A:prepend` / `A:remove` | when A is read, after everything else | no automatic space; order in the file does not matter |
| `A:mymachine = "x"` | when A is read | replaces A if `mymachine` is in OVERRIDES (later entries in OVERRIDES win) |
| `A:append:mymachine = "x"` | when A is read | conditional append |

Read order for a variable: base value (or weak default) -> override replacement -> `:append` / `:prepend` -> `:remove`
-> expand `${...}`. Check real builds with `bitbake-getvar -r <recipe> VAR`.

## Task 1: a mini BitBake variable evaluator
Support exactly the table above (only `=` on override-qualified names like `A:foo`). Lines look like
`NAME OP "value"`; ignore blank lines and `#` comments. Return `{name: final_value}`. For `:remove`, join the remaining
words with single spaces (real BitBake keeps the original whitespace).

In [ ]:
import re

LINE = re.compile(r'^\s*([A-Za-z0-9_\-:]+)\s*(\?\?=|\?=|:=|\+=|=\+|\.=|=\.|=)\s*"(.*)"\s*$')
REF = re.compile(r"\$\{([A-Za-z0-9_\-]+)\}")

def evaluate(text, overrides=()):
    raise NotImplementedError  # TODO

lab.check("evaluator", evaluate)

## Task 2: the task chain and a first recipe
Order the per-recipe tasks, then write a recipe that builds and installs a one-file C program.

In [ ]:
task_chain = None  # TODO: in execution order
lab.check("tasks", task_chain)

In [ ]:
%%writefile hello-rdl_1.0.bb
SUMMARY = "Hello from the robotics daily labs"
# TODO: LICENSE, LIC_FILES_CHKSUM (MIT from COMMON_LICENSE_DIR), SRC_URI with file://hello.c, S,
#       do_compile using ${CC}, ${CFLAGS} and ${LDFLAGS}, do_install into ${D}${bindir}

In [ ]:
lab.check("recipe")

## Checkpoint: Math
**M1 (Remember).** How many tasks are in the chain from `do_fetch` to `do_package` above?

**M2 (Analyze).** `IMAGE_INSTALL = "packagegroup-core-boot"` then `IMAGE_INSTALL:append = "htop"` (no leading space).
How many whitespace-separated words does the final value have?

**M3 (Apply).** `IMAGE_INSTALL = "a b"` then `IMAGE_INSTALL += "htop"`. Number of words?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** You need an extra kernel config option and a patch to `u-boot` from a vendor BSP layer. Where do you put
the change?
- a) edit the vendor layer's recipe in place
- b) in your own layer, as `u-boot_%.bbappend` (and a kernel `.bbappend` with a config fragment), so vendor updates do not wipe it
- c) in `local.conf`

**K2 (Understand).** Explain the roles of `MACHINE`, `DISTRO` and an image recipe, and which layer each lives in.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r05.h`).
3. In `roboprep/checks/r05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()